# 02. 분할·잘림·시간 추이 분석

01 출력으로 다음을 본다. 분할은 **아래 설정 셀의 변수만 바꿔** 다시 돌리면 된다.

1. 구간(train/val/test)별 클래스 엣지 수·비율, 패턴별 시도 수
2. 구간 경계에서 잘리는 시도 — 한 시도가 둘 이상 구간에 걸치는 것 (패턴별)
3. `DATA_END` 이후를 뺄 때 끝이 잘리는 시도 (패턴별), 클래스별 마지막 거래일
4. 클래스별 일별 추이, 패턴별 시도 시작·길이
5. 여러 분할 후보 비교

**잘림이 왜 중요한가** (입력 = 의심 거래만 모은 그래프, 구간 끝까지 누적)
- 학습 엣지의 시도가 검증·테스트로 이어지면 → 학습 그래프에서는 그 시도의 뒷부분이 안 보인다 (잘린 맥락으로 학습)
- 테스트 엣지의 시도가 학습 구간에서 시작했으면 → 같은 시도의 앞부분을 학습에서 봤다 (누수 후보)
- 둘 다 막을 수는 없고, 얼마나 되는지 알아야 지표를 해석할 수 있다

## 0. 설정

In [ ]:
import sys
from pathlib import Path
import pandas as pd
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)

sys.path.append(str(Path.cwd()))
import mc_lib as M

DATASET = "HI-Large"

# 분할 — 양끝 날짜 포함. 기본값은 이진 실험과 같음 (M.WINDOWS)
TRAIN = ("2022-08-08", "2022-10-02")
VAL   = ("2022-10-03", "2022-10-16")
TEST  = ("2022-10-17", "2022-10-30")

DATA_END   = "2022-10-31"     # 이 날짜까지만 쓴다 (11월 이후 제외)
NORMAL_END = "2022-11-05"     # 정상 거래 마지막 날 (01 summary 의 last_day_normal 로 확인)

windows = M.make_windows(TRAIN, VAL, TEST)
OUT = M.results_dir(DATASET, "02_split_analysis")
windows

In [ ]:
lau = M.load_laundering(DATASET)          # 세탁 거래 (엣지 단위, 라벨)
att = M.load_attempts(DATASET)            # 시도 단위
members = M.load_members(DATASET)         # 패턴 파일 행 (시도 × 거래)
daily = M.load_daily(DATASET)             # 일별 전체·세탁·정상 거래 수
print(f"세탁 엣지 {len(lau):,}  시도 {len(att):,}  패턴 행 {len(members):,}")
print("정상 거래 마지막 날:", daily.index[daily['n_normal'] > 0].max().date(), " / 세탁 마지막 날:", lau['ts'].max())

## 1. 구간별 클래스 수

- `edges` : 클래스 × 구간 엣지 수 (라벨 기준 1거래 1행). before = 학습 시작 전, after = 테스트 끝 뒤(11월 꼬리 포함)
- `edges_share` : 각 구간 안에서 클래스 비율(%) — 구간마다 분포가 크게 다르면 검증·테스트 지표 해석에 주의
- `attempts_start` : 시도가 **시작한** 구간 기준 시도 수
- `attempts_touch` : 그 구간에 거래가 1건이라도 있는 시도 수 (한 시도가 여러 열에 셈)

In [ ]:
sc = M.split_counts(lau, members, att, windows)
sc["edges"]

In [ ]:
sc["edges_share"]

In [ ]:
sc["attempts_start"]

In [ ]:
sc["attempts_touch"]

## 2. 구간 경계에서 잘리는 시도

- `cut` : 둘 이상 구간에 걸친 시도. `before|train` … `test|after` : 그 두 구간 모두에 거래가 있는 시도 수
- `outside_share_med(cut)` : 걸친 시도에서 첫 구간 밖으로 나간 행 비율의 중앙값 (클수록 크게 잘림)
- `edge_exposure` : 구간별 패턴 엣지 중 같은 시도가 앞 구간에도 있는 비율(`with_earlier_%`), 뒤 구간에도 있는 비율(`with_later_%`)
  - train 의 `with_earlier_%` 는 before 구간 거래가 학습 그래프의 맥락으로 들어가므로 문제 아님
  - **test 의 `with_earlier_%` = 누수 후보**, **train 의 `with_later_%` = 잘린 맥락으로 학습하는 비율**

In [ ]:
bc = M.boundary_cuts(members, windows)
bc["by_pattern"]

In [ ]:
bc["edge_exposure"].unstack(0)

## 3. `DATA_END` 이후를 뺄 때

- `complete` : DATA_END 안에서 끝난 시도 / `truncated` : 안에서 시작해 밖으로 이어진 시도 / `started_after` : 밖에서 시작한 시도
- `rows_after` : 잘려 나가는 패턴 행 수, `lost_share_*` : 잘린 시도가 잃는 행 비율
- `edges` : 클래스별 DATA_END 까지 / 이후 엣지 수 — **UNMATCHED 의 after_end 가 0 인지** 확인
- `last_seen` : 클래스별 첫·마지막 거래 시각

In [ ]:
tr = M.truncation(members, att, lau, DATA_END)
tr["attempts"]

In [ ]:
tr["edges"]

In [ ]:
tr["last_seen"]

In [ ]:
# 정상 거래가 끝나는 날(NORMAL_END) 기준으로도 — 그 뒤에 새로 시작하는 시도가 있나 (started_after)
tr_n = M.truncation(members, att, lau, NORMAL_END)
tr_n["attempts"][["attempts", "complete", "truncated", "started_after", "truncated_%", "rows_after"]]

In [ ]:
# 패턴별 시도 길이(일)·크기(행 수) — 길수록 경계에 잘리기 쉽다
M.duration_table(att)

## 4. 시간 추이

음영 = train(파랑) / val(주황) / test(초록), 빨간 점선 = DATA_END. 그림 라벨은 영문 (서버 한글 폰트 문제)

In [ ]:
fig = M.plot_daily_classes(lau, windows, DATA_END, daily, path=OUT / "daily_classes.png")

In [ ]:
fig = M.plot_attempt_starts(att, windows, DATA_END, path=OUT / "attempt_starts.png")

## 5. 분할 후보 비교

변수만 바꿔 여러 분할을 한 표로 본다. `min_<split>` = 그 구간에서 가장 적은 클래스의 엣지 수, `test_with_earlier%` = 누수 후보 비율

In [ ]:
configs = {
    "binary_same":   (TRAIN, VAL, TEST),
    # 예시 — 필요하면 바꿔서 추가
    "longer_test":   (("2022-08-08", "2022-09-25"), ("2022-09-26", "2022-10-09"), ("2022-10-10", "2022-10-30")),
}
cmp = M.compare_splits(configs, lau, members, att, DATA_END)
cmp

## 6. 저장

In [ ]:
for name, df in {"edges": sc["edges"], "edges_share": sc["edges_share"], "attempts_start": sc["attempts_start"],
                 "attempts_touch": sc["attempts_touch"], "boundary_cuts": bc["by_pattern"],
                 "edge_exposure": bc["edge_exposure"], "truncation": tr["attempts"], "truncation_edges": tr["edges"],
                 "truncation_normal_end": tr_n["attempts"], "last_seen": tr["last_seen"],
                 "duration": M.duration_table(att), "compare_splits": cmp}.items():
    df.to_csv(OUT / f"{name}.csv", encoding="utf-8-sig")
sorted(p.name for p in OUT.iterdir())